# 09 — Final Evaluation: Ablation Table + Held-Out Test Set

Modeling/tuning is done: `exp07` (Model 2 tuned -- AlephBERT + frozen ResNet50) is the final model, the only one in the project to beat `text_only_bert` on PR-AUC (0.240 vs 0.212 on val). `exp08` (augmentation + re-unfrozen layer4) was tried as one more push and did not beat it (0.222).

This notebook does the two remaining official steps before the report:

1. **Ablation table** (`run_ablation()`, `src/evaluation/metrics.py`) -- assembles the title/image/title+image/full_model comparison from results already on disk (baselines + exp07), answering the project's core question: does the image add value beyond the title?
2. **Test-set evaluation** (`src/evaluation/evaluate_test.py`) -- exp07's checkpoint evaluated on `test.csv` (June 2026), the first and only time the test set is touched in this project.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd

from src.utils import load_config
from src.evaluation.metrics import run_ablation

## 1. Ablation table

Reads `experiments/baselines_results.csv` (title_only/image_only/title_image baselines) + exp07's `val_log.csv` (full_model) fresh -- no hardcoded numbers. Note: no baseline/model in this project ever used tags as input, so `title_tags` is really title-only text (`text_only_bert`) and `full_model` is title+image+site; the function prints a note about this rather than silently relabeling.

In [ ]:
cfg = load_config("../configs/exp07_model2_hybrid_tuned.yaml")
cfg["logging"]["output_dir"] = "../" + cfg["logging"]["output_dir"]  # cwd here is notebooks/, run_ablation assumes project root
ablation_table = run_ablation(cfg)
ablation_table

## 2. Test-set evaluation

**First and only use of `test.csv` in this project.** Loads exp07's `best_model.pt` and evaluates on the held-out June 2026 test set.

In [ ]:
!cd .. && python3 -u -m src.evaluation.evaluate_test --config configs/exp07_model2_hybrid_tuned.yaml

## 3. Summary

Paste the test-set numbers printed above here for the final val-vs-test comparison (checking for overfitting to val across the many tuning iterations).

In [ ]:
val_log = pd.read_csv("../experiments/hybrid_model2_tuned/val_log.csv")
best_val = val_log.loc[val_log["pr_auc"].idxmax()]

test_results = {
    "roc_auc": 0.6973,
    "pr_auc": 0.2122,
    "f1": 0.2701,
    "precision_at_50": 0.4000,
    "precision_at_100": 0.3300,
    "precision_at_200": 0.3000,
}

pd.DataFrame([
    {"split": "val (best checkpoint)", **best_val.drop(["step", "epoch"]).to_dict()},
    {"split": "test (held-out)", **test_results},
])